# Multi-Confirmation Forex / Crypto Trading Bot

**Strategy:** EMA Crossover + ADX Filter + ATR-based SL/TP + Multi-Timeframe Confirmation

**Supports:** Binance · Kraken · Forex (OANDA)

---
### How to use
1. Edit the **⚙️ Configuration** cell below
2. Run all cells top to bottom (`Cell → Run All`)
3. The backtest chart and stats will appear at the bottom
4. When ready for live trading, run the last cell

## 📦 Install Dependencies

In [ ]:
# Run once to install required libraries
!pip install pandas numpy matplotlib python-dotenv ccxt oandapyV20 -q

## ⚙️ Configuration
**This is the only cell you need to edit.** Change broker, symbol, timeframes and strategy parameters here.

In [ ]:
import os

# ── Broker ────────────────────────────────────────────────────
# Options: "binance" | "kraken" | "forex"
BROKER = os.getenv("BROKER", "binance")

# ── Asset ─────────────────────────────────────────────────────
# Binance / Kraken : "BTC/USDT"  "ETH/USDT"  "SOL/USDT"
# Forex (OANDA)    : "EUR_USD"   "GBP_USD"   "USD_JPY"
SYMBOL = os.getenv("SYMBOL", "BTC/USDT")

# ── Timeframes ────────────────────────────────────────────────
# Options : "1m" "5m" "15m" "30m" "1h" "4h" "1d"
# Rule    : HTF should always be one step above LTF
LTF = os.getenv("LTF", "1h")   # Trading timeframe
HTF = os.getenv("HTF", "4h")   # Higher timeframe for bias

# ── EMA Crossover ─────────────────────────────────────────────
EMA_FAST = int(os.getenv("EMA_FAST", "9"))
EMA_SLOW = int(os.getenv("EMA_SLOW", "21"))

# ── ADX (trend strength filter) ───────────────────────────────
ADX_PERIOD    = int(os.getenv("ADX_PERIOD", "14"))
ADX_THRESHOLD = float(os.getenv("ADX_THRESHOLD", "25"))  # Only trade when ADX > this

# ── ATR (volatility filter + dynamic SL/TP) ───────────────────
ATR_PERIOD  = int(os.getenv("ATR_PERIOD", "14"))
ATR_SL_MULT = float(os.getenv("ATR_SL_MULT", "1.5"))   # Stop loss  = entry ± 1.5×ATR
ATR_TP_MULT = float(os.getenv("ATR_TP_MULT", "2.5"))   # Take profit = entry ± 2.5×ATR

# ── Risk & Capital ────────────────────────────────────────────
RISK_PCT        = float(os.getenv("RISK_PCT", "1.0"))          # % equity risked per trade
INITIAL_CAPITAL = float(os.getenv("INITIAL_CAPITAL", "10000")) # Starting capital ($)
BACKTEST_DAYS   = int(os.getenv("BACKTEST_DAYS", "365"))        # Days of history to test

# ── API Credentials ───────────────────────────────────────────
# Leave blank to run in demo mode with synthetic data
BINANCE_API_KEY    = os.getenv("BINANCE_API_KEY", "")
BINANCE_API_SECRET = os.getenv("BINANCE_API_SECRET", "")

KRAKEN_API_KEY     = os.getenv("KRAKEN_API_KEY", "")
KRAKEN_API_SECRET  = os.getenv("KRAKEN_API_SECRET", "")

OANDA_API_KEY      = os.getenv("OANDA_API_KEY", "")
OANDA_ACCOUNT_ID   = os.getenv("OANDA_ACCOUNT_ID", "")
OANDA_ENVIRONMENT  = os.getenv("OANDA_ENVIRONMENT", "practice")  # "practice" | "live"

print(f"✅ Config loaded — Broker: {BROKER} | Symbol: {SYMBOL} | LTF: {LTF} | HTF: {HTF}")

## 📥 Data Fetchers

In [ ]:
import warnings
import logging
import time
from datetime import datetime, timedelta

import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s",
                    datefmt="%H:%M:%S")
log = logging.getLogger(__name__)


def fetch_binance(symbol, timeframe, limit=1000):
    import ccxt
    exchange = ccxt.binance({"apiKey": BINANCE_API_KEY, "secret": BINANCE_API_SECRET})
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
    df = pd.DataFrame(ohlcv, columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df.set_index("timestamp")


def fetch_kraken(symbol, timeframe, limit=1000):
    import ccxt
    exchange = ccxt.kraken({"apiKey": KRAKEN_API_KEY, "secret": KRAKEN_API_SECRET})
    ohlcv = exchange.fetch_ohlcv(symbol, timeframe, limit=limit)
    df = pd.DataFrame(ohlcv, columns=["timestamp","open","high","low","close","volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    return df.set_index("timestamp")


def fetch_forex(symbol, timeframe, limit=1000):
    import oandapyV20
    import oandapyV20.endpoints.instruments as instruments
    tf_map = {"1m":"M1","5m":"M5","15m":"M15","30m":"M30","1h":"H1","4h":"H4","1d":"D"}
    client = oandapyV20.API(access_token=OANDA_API_KEY, environment=OANDA_ENVIRONMENT)
    r = instruments.InstrumentsCandles(instrument=symbol,
                                        params={"granularity": tf_map.get(timeframe,"H1"),
                                                "count": limit})
    client.request(r)
    rows = [{"timestamp": pd.to_datetime(c["time"]),
             "open":  float(c["mid"]["o"]), "high":  float(c["mid"]["h"]),
             "low":   float(c["mid"]["l"]), "close": float(c["mid"]["c"]),
             "volume": int(c["volume"])} for c in r.response["candles"] if c["complete"]]
    return pd.DataFrame(rows).set_index("timestamp")


def fetch_demo_data(symbol, timeframe, days=365):
    """Synthetic OHLCV data with realistic trend regimes for demo/testing."""
    tf_minutes = {"1m":1,"5m":5,"15m":15,"30m":30,"1h":60,"4h":240,"1d":1440}
    mins = tf_minutes.get(timeframe, 60)
    n = (days * 1440) // mins
    np.random.seed(42)
    log_returns = np.random.normal(0.0001, 0.002, n)
    for i in range(0, n, n // 6):
        log_returns[i:i + n//6] += np.random.choice([-1,1]) * 0.0003
    prices = 100 * np.exp(np.cumsum(log_returns))
    high   = prices * (1 + np.abs(np.random.normal(0, 0.003, n)))
    low    = prices * (1 - np.abs(np.random.normal(0, 0.003, n)))
    open_  = np.roll(prices, 1); open_[0] = prices[0]
    idx    = pd.date_range(end=datetime.now(), periods=n, freq=f"{mins}min")
    return pd.DataFrame({"open":open_,"high":high,"low":low,
                          "close":prices,"volume":np.random.randint(1000,50000,n).astype(float)}, index=idx)


def get_data(symbol, timeframe, days=365):
    tf_minutes = {"1m":1,"5m":5,"15m":15,"30m":30,"1h":60,"4h":240,"1d":1440}
    limit = max(500, (days * 1440) // tf_minutes.get(timeframe, 60))
    has_creds = {"binance": bool(BINANCE_API_KEY), "kraken": bool(KRAKEN_API_KEY),
                 "forex":   bool(OANDA_API_KEY)}
    if not has_creds.get(BROKER.lower(), False):
        log.warning(f"No API credentials — using synthetic demo data")
        return fetch_demo_data(symbol, timeframe, days)
    return {"binance": fetch_binance, "kraken": fetch_kraken,
            "forex": fetch_forex}[BROKER.lower()](symbol, timeframe, limit)


print("✅ Data fetchers ready")

## 📐 Indicators
EMA · ATR · ADX

In [ ]:
def ema(series, period):
    return series.ewm(span=period, adjust=False).mean()


def calc_atr(df, period=14):
    hl = df["high"] - df["low"]
    hc = (df["high"] - df["close"].shift()).abs()
    lc = (df["low"]  - df["close"].shift()).abs()
    tr = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.ewm(span=period, adjust=False).mean()


def calc_adx(df, period=14):
    up   = df["high"].diff()
    down = -df["low"].diff()
    plus_dm  = np.where((up > down) & (up > 0), up, 0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)
    tr_val   = calc_atr(df, period)
    plus_di  = 100 * pd.Series(plus_dm,  index=df.index).ewm(span=period, adjust=False).mean() / tr_val
    minus_di = 100 * pd.Series(minus_dm, index=df.index).ewm(span=period, adjust=False).mean() / tr_val
    dx = (100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan))
    return dx.ewm(span=period, adjust=False).mean()


def compute_indicators(df):
    df = df.copy()
    df["ema_fast"] = ema(df["close"], EMA_FAST)
    df["ema_slow"] = ema(df["close"], EMA_SLOW)
    df["atr"]      = calc_atr(df, ATR_PERIOD)
    df["atr_ma"]   = df["atr"].rolling(ATR_PERIOD * 2).mean()
    df["adx"]      = calc_adx(df, ADX_PERIOD)
    return df.dropna()


print("✅ Indicators ready")

## 🎯 Signal Generation
All 5 confirmation layers must agree before a trade is taken:

| Layer | Filter | Purpose |
|---|---|---|
| 1 | HTF EMA bias | Only trade in the direction of the higher timeframe trend |
| 2 | LTF EMA crossover | Entry timing signal |
| 3 | ADX > threshold | Confirm a real trend exists (not ranging) |
| 4 | ATR > ATR MA | Confirm sufficient volatility to produce a real move |
| 5 | ATR-based SL/TP | Dynamic stop loss and take profit sized to current volatility |

In [ ]:
def generate_signals(ltf_df, htf_df):
    df = ltf_df.copy()

    # Layer 1 — HTF trend bias (resampled to LTF index)
    htf_trend = (htf_df["ema_fast"] > htf_df["ema_slow"]).astype(int)
    htf_trend = htf_trend.reindex(df.index, method="ffill")

    # Layer 2 — LTF EMA crossover
    cross_up   = (df["ema_fast"] > df["ema_slow"]) & (df["ema_fast"].shift() <= df["ema_slow"].shift())
    cross_down = (df["ema_fast"] < df["ema_slow"]) & (df["ema_fast"].shift() >= df["ema_slow"].shift())

    # Layer 3 — ADX trend strength
    trend_ok = df["adx"] > ADX_THRESHOLD

    # Layer 4 — ATR volatility
    vol_ok = df["atr"] > df["atr_ma"]

    # Combine all layers
    df["signal"] = 0
    df.loc[cross_up   & trend_ok & vol_ok & (htf_trend == 1), "signal"] =  1  # BUY
    df.loc[cross_down & trend_ok & vol_ok & (htf_trend == 0), "signal"] = -1  # SELL

    # Layer 5 — ATR-based SL / TP
    df["sl"] = np.where(df["signal"] ==  1, df["close"] - ATR_SL_MULT * df["atr"],
               np.where(df["signal"] == -1, df["close"] + ATR_SL_MULT * df["atr"], np.nan))
    df["tp"] = np.where(df["signal"] ==  1, df["close"] + ATR_TP_MULT * df["atr"],
               np.where(df["signal"] == -1, df["close"] - ATR_TP_MULT * df["atr"], np.nan))
    return df


print("✅ Signal generator ready")

## 🔁 Backtester

In [ ]:
class Backtester:
    def __init__(self, df):
        self.df     = df
        self.equity = INITIAL_CAPITAL
        self.trades = []
        self.equity_curve = []

    def run(self):
        position = None
        entry_price = sl = tp = entry_time = 0

        for ts, row in self.df.iterrows():
            self.equity_curve.append({"timestamp": ts, "equity": self.equity})

            # ── Check exit ────────────────────────────
            if position is not None:
                hit_sl = (position ==  1 and row["low"]  <= sl) or \
                         (position == -1 and row["high"] >= sl)
                hit_tp = (position ==  1 and row["high"] >= tp) or \
                         (position == -1 and row["low"]  <= tp)

                if hit_sl or hit_tp:
                    exit_price = sl if hit_sl else tp
                    pnl_pct    = (exit_price - entry_price) / entry_price * position
                    risk_base  = ATR_SL_MULT * row["atr"] / entry_price
                    pnl_dollar = self.equity * RISK_PCT / 100 * pnl_pct / max(risk_base, 1e-9)
                    self.equity += pnl_dollar
                    self.trades.append({
                        "entry_time":  entry_time,  "exit_time":   ts,
                        "direction":   "LONG" if position == 1 else "SHORT",
                        "entry_price": entry_price, "exit_price":  exit_price,
                        "sl":          sl,           "tp":          tp,
                        "pnl":         pnl_dollar,
                        "result":      "WIN" if hit_tp else "LOSS",
                    })
                    position = None

            # ── New entry ─────────────────────────────
            if position is None and row["signal"] != 0:
                position    = row["signal"]
                entry_price = row["close"]
                entry_time  = ts
                sl          = row["sl"]
                tp          = row["tp"]

        return self._stats()

    def _stats(self):
        if not self.trades:
            return {"error": "No trades generated — try loosening ADX_THRESHOLD or extending BACKTEST_DAYS"}

        trades_df  = pd.DataFrame(self.trades)
        wins       = trades_df[trades_df["result"] == "WIN"]
        losses     = trades_df[trades_df["result"] == "LOSS"]
        equity_df  = pd.DataFrame(self.equity_curve).set_index("timestamp")
        drawdown   = (equity_df["equity"] - equity_df["equity"].cummax()) / equity_df["equity"].cummax() * 100
        returns    = trades_df["pnl"] / INITIAL_CAPITAL
        sharpe     = (returns.mean() / returns.std() * np.sqrt(252)) if returns.std() > 0 else 0

        return {
            "total_trades":  len(trades_df),
            "win_rate":      round(len(wins) / len(trades_df) * 100, 2),
            "total_return":  round((self.equity - INITIAL_CAPITAL) / INITIAL_CAPITAL * 100, 2),
            "final_equity":  round(self.equity, 2),
            "sharpe_ratio":  round(sharpe, 2),
            "max_drawdown":  round(drawdown.min(), 2),
            "avg_win":       round(wins["pnl"].mean(), 2)   if len(wins)   > 0 else 0,
            "avg_loss":      round(losses["pnl"].mean(), 2) if len(losses) > 0 else 0,
            "profit_factor": round(wins["pnl"].sum() / abs(losses["pnl"].sum()), 2) if len(losses) > 0 else float("inf"),
            "trades_df":     trades_df,
            "equity_curve":  equity_df,
        }


print("✅ Backtester ready")

## 🚀 Run Backtest

In [ ]:
log.info(f"Fetching {SYMBOL} data — LTF: {LTF} | HTF: {HTF} | {BACKTEST_DAYS} days")

ltf_raw  = get_data(SYMBOL, LTF, days=BACKTEST_DAYS)
htf_raw  = get_data(SYMBOL, HTF, days=BACKTEST_DAYS * 2)

ltf_ind  = compute_indicators(ltf_raw)
htf_ind  = compute_indicators(htf_raw)

sig_df   = generate_signals(ltf_ind, htf_ind)

bt       = Backtester(sig_df)
stats    = bt.run()

if "error" in stats:
    print(f"⚠️  {stats['error']}")
else:
    print(f"""
╔══════════════════════════════════════╗
║         BACKTEST RESULTS             ║
╠══════════════════════════════════════╣
║  Symbol        : {SYMBOL:<20}║
║  Timeframe     : {LTF} → {HTF:<15}║
╠══════════════════════════════════════╣
║  Total trades  : {stats['total_trades']:<20}║
║  Win rate      : {stats['win_rate']:<19}%║
║  Total return  : {stats['total_return']:<19}%║
║  Final equity  : ${stats['final_equity']:<19,}║
║  Sharpe ratio  : {stats['sharpe_ratio']:<20}║
║  Max drawdown  : {stats['max_drawdown']:<19}%║
║  Profit factor : {stats['profit_factor']:<20}║
║  Avg win       : ${stats['avg_win']:<19}║
║  Avg loss      : ${stats['avg_loss']:<19}║
╚══════════════════════════════════════╝
""")

## 📊 Trade Log

In [ ]:
if "trades_df" in stats:
    display(stats["trades_df"].style
        .applymap(lambda v: "color: #1D9E75" if v == "WIN" else "color: #D85A30",
                  subset=["result"])
        .format({"entry_price": "{:.4f}", "exit_price": "{:.4f}",
                 "sl": "{:.4f}", "tp": "{:.4f}", "pnl": "${:.2f}"})
    )

## 📈 Backtest Chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline
plt.style.use("dark_background")

if "error" not in stats:
    trades    = stats["trades_df"]
    equity    = stats["equity_curve"]

    fig = plt.figure(figsize=(16, 12), facecolor="#0f0f0f")
    gs  = gridspec.GridSpec(4, 1, figure=fig, hspace=0.06, height_ratios=[3, 1, 1, 1.5])
    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax4 = fig.add_subplot(gs[3])

    for ax in [ax1, ax2, ax3, ax4]:
        ax.set_facecolor("#0f0f0f")
        ax.tick_params(colors="#666", labelsize=8)
        for spine in ax.spines.values(): spine.set_color("#333")

    # Price + EMAs
    ax1.plot(sig_df.index, sig_df["close"],    color="#555",   lw=0.8, label="Price")
    ax1.plot(sig_df.index, sig_df["ema_fast"], color="#378ADD", lw=1.2, label=f"EMA {EMA_FAST}")
    ax1.plot(sig_df.index, sig_df["ema_slow"], color="#E85D24", lw=1.2, label=f"EMA {EMA_SLOW}")

    for _, t in trades.iterrows():
        c = "#1D9E75" if t["direction"] == "LONG" else "#D85A30"
        m = "^" if t["direction"] == "LONG" else "v"
        ax1.scatter(t["entry_time"], t["entry_price"], color=c,     marker=m,   s=70, zorder=5)
        ax1.scatter(t["exit_time"],  t["exit_price"],  color="#aaa", marker="x", s=40, zorder=5)

    ax1.set_ylabel("Price", color="#888", fontsize=9)
    ax1.legend(loc="upper left", fontsize=8, facecolor="#1a1a1a", labelcolor="white", framealpha=0.7)
    ax1.set_title(f"{SYMBOL} {LTF} | Return: {stats['total_return']}%  "
                  f"Win Rate: {stats['win_rate']}%  Sharpe: {stats['sharpe_ratio']}  "
                  f"Max DD: {stats['max_drawdown']}%",
                  color="white", fontsize=10, pad=8)

    # ADX
    ax2.plot(sig_df.index, sig_df["adx"], color="#9F7FD4", lw=1)
    ax2.axhline(ADX_THRESHOLD, color="#555", lw=0.8, linestyle="--")
    ax2.set_ylabel("ADX", color="#888", fontsize=9)
    ax2.set_ylim(0, 60)

    # ATR
    ax3.plot(sig_df.index, sig_df["atr"],    color="#E8B84B", lw=1,   label="ATR")
    ax3.plot(sig_df.index, sig_df["atr_ma"], color="#555",    lw=0.8, linestyle="--", label="ATR MA")
    ax3.set_ylabel("ATR", color="#888", fontsize=9)
    ax3.legend(loc="upper left", fontsize=7, facecolor="#1a1a1a", labelcolor="white", framealpha=0.6)

    # Equity curve
    eq_color = "#1D9E75" if stats["total_return"] >= 0 else "#D85A30"
    ax4.plot(equity.index, equity["equity"], color=eq_color, lw=1.5)
    ax4.fill_between(equity.index, equity["equity"], INITIAL_CAPITAL, alpha=0.15, color=eq_color)
    ax4.axhline(INITIAL_CAPITAL, color="#555", lw=0.8, linestyle="--")
    ax4.set_ylabel("Equity ($)", color="#888", fontsize=9)
    ax4.set_xlabel("Date", color="#888", fontsize=9)

    plt.tight_layout()
    plt.savefig(f"backtest_{SYMBOL.replace('/','-')}_{LTF}.png", dpi=150,
                bbox_inches="tight", facecolor="#0f0f0f")
    plt.show()
    print("📁 Chart also saved as PNG in the current directory")

## 💾 Export Trade Log to CSV

In [ ]:
if "trades_df" in stats:
    filename = f"trades_{SYMBOL.replace('/','-')}_{LTF}.csv"
    stats["trades_df"].to_csv(filename, index=False)
    print(f"✅ Trade log saved → {filename}")

## ⚡ Live Trading

> ⚠️ **Make sure your API credentials are set in the Configuration cell before running this.**
> Start with paper trading / practice accounts first.

In [ ]:
class LiveTrader:
    """
    Live trading loop.
    Polls for new candles on each LTF close and fires orders
    when all 5 confirmation layers align.
    """
    def __init__(self):
        self.position = None

    def place_order(self, direction, entry, sl, tp):
        """
        Connect your broker's order API here.

        Binance / Kraken (ccxt):
            exchange.create_order(SYMBOL, 'market', 'buy', qty)
            exchange.create_order(SYMBOL, 'stop',   'sell', qty, sl)
            exchange.create_order(SYMBOL, 'limit',  'sell', qty, tp)

        Forex (oandapyV20):
            import oandapyV20.endpoints.orders as orders
            data = {"order": {"type": "MARKET", "instrument": SYMBOL, "units": "100",
                    "stopLossOnFill": {"price": str(sl)},
                    "takeProfitOnFill": {"price": str(tp)}}}
            r = orders.OrderCreate(OANDA_ACCOUNT_ID, data=data)
            client.request(r)
        """
        log.info(f"🟢 ORDER  {direction}  entry={entry:.5f}  SL={sl:.5f}  TP={tp:.5f}")

    def run(self, max_iterations=None):
        tf_seconds = {"1m":60,"5m":300,"15m":900,"30m":1800,
                      "1h":3600,"4h":14400,"1d":86400}
        sleep_time = tf_seconds.get(LTF, 3600)
        i = 0
        log.info(f"🚀 Live trader started — {BROKER} | {SYMBOL} | {LTF}")

        while True:
            try:
                ltf_df  = get_data(SYMBOL, LTF,  days=30)
                htf_df  = get_data(SYMBOL, HTF,  days=60)
                ltf_ind = compute_indicators(ltf_df)
                htf_ind = compute_indicators(htf_df)
                sig     = generate_signals(ltf_ind, htf_ind)
                latest  = sig.iloc[-1]

                log.info(f"Signal: {latest['signal']}  ADX: {latest['adx']:.1f}  "
                         f"ATR: {latest['atr']:.5f}  Price: {latest['close']:.5f}")

                if latest["signal"] != 0 and self.position is None:
                    direction = "LONG" if latest["signal"] == 1 else "SHORT"
                    self.place_order(direction, latest["close"], latest["sl"], latest["tp"])
                    self.position = latest["signal"]

            except KeyboardInterrupt:
                log.info("⛔ Live trader stopped")
                break
            except Exception as e:
                log.error(f"Loop error: {e}")

            i += 1
            if max_iterations and i >= max_iterations:
                break
            time.sleep(sleep_time)


# ── Uncomment the line below to start live trading ───────────
# LiveTrader().run()

print("✅ LiveTrader defined — uncomment the last line to start")